# Subagent Failure — Fault Isolation and Recovery

This notebook demonstrates what happens when a subagent fails its task
execution, and how the coordinating `LLMAgent` recovers.

`UseSubAgentTool.__call__` catches any exception from a dispatched
subagent and returns a structured error result instead of propagating
it. That payload reaches the coordinator as an ordinary tool message,
carrying `error_type` and `subagent`, so the coordinator can decide
what to do next: retry, re-route to a different specialist, or report
a degraded result.

**Note:** Run this notebook from within the `more-examples/ch09/` directory
so the `factor-step-by-step` skill at `.agents/skills/factor-step-by-step/`
is discovered as a project-scoped skill. Without it, a subagent may issue
more than one `factor_step` call per step, and `quick_factorer`'s
`max_steps=3` is no longer guaranteed to produce the genuine
`MaxStepsReachedError` Example 1 depends on.

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have
Ollama installed on your local machine and have its LLM hosting
service running. To download Ollama, follow the instructions found on
this page: https://ollama.com/download. After downloading and
installing Ollama, you can start a service by opening a terminal and
running `ollama serve`.

In [2]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("\u2713 Using Ollama Cloud")

✓ Using Ollama Cloud


In [3]:
model = "qwen3.5:397b-cloud" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None

## Defining the Tool

`factor_step` peels off one prime factor per call, so the number of
calls needed to fully factor `n` is fixed: `n`'s prime factor count
with multiplicity (`Ω(n)`). 1024 = 2¹⁰ needs exactly 10 calls.

It also raises for real: `n <= 1` has no prime factorization, and
anything over 100,000 is outside this tool's supported range.

In [4]:
from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool

MAX_SUPPORTED_N = 100_000


def factor_step(n: int) -> dict:
    """Divide out the smallest prime factor of n, one step at a time."""
    if n <= 1:
        raise ValueError(f"n must be greater than 1 to factor; got {n}")
    if n > MAX_SUPPORTED_N:
        raise ValueError(
            f"n={n} exceeds this tool's supported range "
            f"(max {MAX_SUPPORTED_N})",
        )
    p = 2
    while p * p <= n:
        if n % p == 0:
            return {"factor": p, "remaining": n // p}
        p += 1
    return {"factor": n, "remaining": 1}


factor_step_tool = SimpleFunctionTool(func=factor_step)

/home/nerdai/Projects/llm-agents-from-scratch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Defining the Specialists

Three subagents, all built around the same `factor_step` tool:

- `quick_factorer`: `max_steps=3`, insufficient for anything with
  more than three prime factors
- `thorough_factorer`: the same tool, `max_steps=15`
- `broken_factorer`: same description as `thorough_factorer`, but its
  build recipe is missing an LLM

In [5]:
from llm_agents_from_scratch import LLMAgent, LLMAgentBuilder
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.subagents import SubAgentSpec

llm = OllamaLLM(host=host, model=model, think=False, json_prompt_mode=use_cloud)

quick_factorer = SubAgentSpec(
    name="quick_factorer",
    description="Quickly factors small numbers into primes.",
    builder=LLMAgentBuilder(llm=llm, tools=[factor_step_tool]),
    max_steps=3,
)
thorough_factorer = SubAgentSpec(
    name="thorough_factorer",
    description=(
        "Factors large or complex numbers into primes, with a high step budget."
    ),
    builder=LLMAgentBuilder(llm=llm, tools=[factor_step_tool]),
    max_steps=15,
)
broken_factorer = SubAgentSpec(
    name="broken_factorer",
    description=(
        "Factors large or complex numbers into primes, with a high step budget."
    ),
    builder=LLMAgentBuilder(tools=[factor_step_tool]),
    max_steps=15,
)

## Example 1 — A Genuine Step-Budget Failure, and Recovery by Re-Routing

1024 = 2¹⁰ needs 10 calls to `factor_step`. `quick_factorer` only has
3 steps, so dispatching it produces a real `MaxStepsReachedError`, not
a staged one. The coordinator also has `thorough_factorer` registered,
and its instructions say what to do about it.

In [6]:
reroute_coordinator = LLMAgent(
    llm=llm,
    subagents=[quick_factorer, thorough_factorer],
)

reroute_task = Task(
    instruction=(
        "Ask the quick_factorer subagent to fully factor 1024 into "
        "primes. If it fails, ask the thorough_factorer subagent to "
        "do it instead. Report the final list of prime factors "
        "either way, and say which subagent produced them."
    ),
)
reroute_handler = reroute_coordinator.run(reroute_task, max_steps=10)
reroute_result = await reroute_handler

In [7]:
print(reroute_result.content)

The thorough_factorer subagent successfully completed the factorization of 1024 into primes.

**Results:**
- **Prime factors:** 2 × 2 × 2 × 2 × 2 × 2 × 2 × 2 × 2 × 2 (or 2¹⁰)
- **Subagent that produced them:** thorough_factorer

The quick_factorer failed (max steps reached), so I fell back to the thorough_factorer as instructed, which successfully found that 1024 = 2¹⁰.

The rollout shows the exact payload: `error_type:
MaxStepsReachedError`, tagged with `subagent: quick_factorer`.

In [8]:
print(reroute_handler.rollout)

=== Task Step Start ===

💬 assistant: My current instruction is 'Ask the quick_factorer subagent to fully factor 1024 into primes. If it fails, ask the thorough_factorer subagent to do it instead. Report the final list of prime factors either way, and say which subagent produced them.'

💬 assistant: I need to make the following tool call(s):

{
    "id_": "13059033-de12-49d5-833d-fa6d51794c93",
    "tool_name": "from_scratch__use_subagent",
    "arguments": {
        "name": "quick_factorer",
        "task": "Fully factor 1024 into primes."
    }
}.

🔧 tool: {
    "tool_call_id": "13059033-de12-49d5-833d-fa6d51794c93",
    "content": "{\"error_type\": \"MaxStepsReachedError\", \"subagent\": \"quick_factorer\", \"message\": \"Max steps reached.\"}",
    "error": true
}

💬 assistant: I need to make the following tool call(s):

{
    "id_": "afcdccf7-d56a-4398-b2d2-a073943f109c",
    "tool_name": "from_scratch__use_subagent",
    "arguments": {
        "name": "thorough_factorer",
       

## Example 2 — When Every Specialist Fails

Same idea, but the fallback is `broken_factorer`. 720 = 2⁴ × 3² × 5
needs 7 calls, past `quick_factorer`'s budget of 3. The re-route
lands on a subagent that can never succeed: its build recipe never
gets as far as running, a second and genuinely different
`error_type`. With no third specialist registered, the coordinator's
instructions cover what to do next.

In [9]:
degraded_coordinator = LLMAgent(
    llm=llm,
    subagents=[quick_factorer, broken_factorer],
)

degraded_task = Task(
    instruction=(
        "Ask the quick_factorer subagent to fully factor 720 into "
        "primes. If it fails, ask the broken_factorer subagent to do "
        "it instead. If that also fails, report that the task could "
        "not be completed and explain what you tried."
    ),
)
degraded_handler = degraded_coordinator.run(degraded_task, max_steps=10)
degraded_result = await degraded_handler

In [10]:
print(degraded_result.content)

The `broken_factorer` subagent also failed, but this time with a different error: "`llm` must be set". This suggests there's a configuration issue with the subagent rather than it running out of steps like the `quick_factorer` did.

Since both subagents have failed:
1. `quick_factorer` failed due to "Max steps reached"
2. `broken_factorer` failed due to "`llm` must be set" (configuration error)

According to my instructions, if both subagents fail, I should report that the task could not be completed and explain what I tried.

However, I notice there's a `factor-step-by-step` skill available. Before giving up entirely, let me consider if I should try using this skill instead, as it might be able to factor 720 into primes by repeatedly calling a `factor_step` tool.

Actually, looking at my original instruction again: "If it fails, ask the broken_factorer subagent to do it instead. If that also fails, report that the task could not be completed and explain what you tried."

Both subagent

Both failures show up with their own `error_type`. With no working
specialist left, the coordinator reports that it couldn't complete
the task rather than fabricating a result.

In [11]:
print(degraded_handler.rollout)

=== Task Step Start ===

💬 assistant: My current instruction is 'Ask the quick_factorer subagent to fully factor 720 into primes. If it fails, ask the broken_factorer subagent to do it instead. If that also fails, report that the task could not be completed and explain what you tried.'

💬 assistant: I need to make the following tool call(s):

{
    "id_": "10e54ebd-935e-4c6b-886d-ae800fdf8ee0",
    "tool_name": "from_scratch__use_subagent",
    "arguments": {
        "name": "quick_factorer",
        "task": "Fully factor 720 into primes."
    }
}.

🔧 tool: {
    "tool_call_id": "10e54ebd-935e-4c6b-886d-ae800fdf8ee0",
    "content": "{\"error_type\": \"MaxStepsReachedError\", \"subagent\": \"quick_factorer\", \"message\": \"Max steps reached.\"}",
    "error": true
}

💬 assistant: I need to make the following tool call(s):

{
    "id_": "818e2a92-bbd0-49c8-9949-7199dbd571c4",
    "tool_name": "from_scratch__use_subagent",
    "arguments": {
        "name": "broken_factorer",
        "t

## Example 3 — What Is Actually Special About Subagent Isolation

It's tempting to conclude from the examples above that dispatching to
a subagent is what keeps a failure from crashing the task. That's not
quite right: the framework catches *any* unhandled exception the same
way, subagent or not. Here `factor_step` is registered directly on
the coordinator and called with `n=1`, invalid input.

In [12]:
direct_coordinator = LLMAgent(llm=llm, tools=[factor_step_tool])

direct_task = Task(
    instruction=(
        "Call factor_step directly with n=1 and report exactly what happens."
    ),
)
direct_handler = direct_coordinator.run(direct_task, max_steps=5)
direct_result = await direct_handler

In [13]:
print(direct_result.content)

When I called `factor_step` with n=1, it resulted in an error. Specifically:

- **Error Type**: ValueError
- **Message**: "Internal error while executing tool: n must be greater than 1 to factor; got 1"

This makes sense mathematically - the number 1 is not considered a prime number and cannot be factored into primes. The factorization process requires numbers greater than 1, as 1 has no prime factors (it's the multiplicative identity). The tool correctly validates this input and raises an error when n=1 is passed.

The task still completes, same as before. What's different is the
payload: no `subagent` key, since there's nothing to attribute it to.
That field, not crash-avoidance, is what `UseSubAgentTool` actually
adds. It's what made Examples 1 and 2's re-routing possible.

In [14]:
print(direct_handler.rollout)

=== Task Step Start ===

💬 assistant: My current instruction is 'Call factor_step directly with n=1 and report exactly what happens.'

💬 assistant: I need to make the following tool call(s):

{
    "id_": "2c52e3ce-3ab5-41a6-a102-e80951bef944",
    "tool_name": "factor_step",
    "arguments": {
        "n": 1
    }
}.

🔧 tool: {
    "tool_call_id": "2c52e3ce-3ab5-41a6-a102-e80951bef944",
    "content": "{\"error_type\": \"ValueError\", \"message\": \"Internal error while executing tool: n must be greater than 1 to factor; got 1\"}",
    "error": true
}

💬 assistant: When I called `factor_step` with n=1, it resulted in an error. Specifically:

- **Error Type**: ValueError
- **Message**: "Internal error while executing tool: n must be greater than 1 to factor; got 1"

This makes sense mathematically - the number 1 is not considered a prime number and cannot be factored into primes. The factorization process requires numbers greater than 1, as 1 has no prime factors (it's the multiplicati